# Stage 12 – Unsupervised Anomaly-Score Investigation
**Project:** DGI-GCN adversarial detection in Alzheimer’s MRI classification

## Purpose
This stage compares three *unsupervised scoring rules* using the same learned representation:
1. k-nearest-neighbour (kNN) distance
2. Regularised Mahalanobis distance
3. Local-neighbour affinity

The DGI-GCN encoder remains frozen. No attack labels are used to fit a detector or choose thresholds. Clean training embeddings define the reference distribution; clean validation data calibrates a threshold at a predeclared false-positive target. The test set is used only once for final evaluation.

**Important:** This notebook is an investigation, not a promise of improved performance. It will report the results even if a method performs poorly. Do not change the test set, threshold rule, or scoring formula after inspecting test results.

## 0. Runtime and reproducibility
Use Google Colab with a GPU runtime if available. Set the random seed and install only the required packages.

In [1]:
import os, random, json, zipfile, glob, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix
from sklearn.neighbors import NearestNeighbors
from google.colab import drive

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


## 1. Mount Drive and point to the existing Stage outputs

Create a folder in Google Drive named `Alzheimer_DGI_Research`. Put the original ZIP files below into that folder (do not rename files inside the ZIPs):

- `Stage5_Graph_Outputs.zip`
- `Stage6_DGI_Outputs.zip`
- `Stage7_Outputs.zip` (the ZIP produced by your corrected Stage 7 notebook)
- `Stage8_Corrected_Outputs.zip`
- `Stage11_CW_L2_Investigation_Outputs.zip`

The notebook will check the files before proceeding. If Stage 7's ZIP has a different filename, change `STAGE7_ZIP` below to its exact name.

**Do not use Stage 8/9/10/11 test labels to fit any scoring method or threshold.** Those outputs are retained only for later audit/comparison.

In [9]:
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/Alzheimer_DGI_Research')
OUT = ROOT / 'Stage12_Outputs'
WORK = Path('/content/stage12_work')
OUT.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

# Keep track of paths (supporting both .zip files and extracted directories)
STAGE5_ZIP = ROOT / 'Stage5_Graph_Outputs.zip'
STAGE6_ZIP = ROOT / 'Stage6_DGI_Outputs.zip'
STAGE7_ZIP = ROOT / 'Stage7_Outputs.zip'
STAGE8_ZIP = ROOT / 'Stage8_Corrected_Outputs.zip'
STAGE11_ZIP = ROOT / 'Stage11_CW_L2_Investigation_Outputs.zip'

STAGE5_DIR = ROOT / 'Stage5_Graph_Outputs'
STAGE6_DIR = ROOT / 'Stage6_DGI_Outputs'
STAGE7_DIR = ROOT / 'Stage7_Outputs'

# Ensure we have either the zip file or the extracted directory
assert STAGE5_ZIP.exists() or STAGE5_DIR.exists(), "Missing Stage 5 outputs (zip or folder)"
assert STAGE6_ZIP.exists() or STAGE6_DIR.exists(), "Missing Stage 6 outputs (zip or folder)"
assert STAGE7_ZIP.exists() or STAGE7_DIR.exists(), "Missing Stage 7 outputs (zip or folder)"

print("Verification successful! Proceeding with available Stage folders/archives.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verification successful! Proceeding with available Stage folders/archives.


In [7]:
import os
from pathlib import Path

# Define the base drive path
drive_path = Path('/content/drive/MyDrive')

if drive_path.exists():
    print("--- Root of Google Drive (My Drive) Contents ---")
    subdirs = []
    for p in sorted(drive_path.glob('*')):
        if p.is_dir():
            subdirs.append(p.name)
            if 'alzheimer' in p.name.lower():
                print(f"[FOLDER] {p.name} (Matched Keyword)")
                for child in sorted(p.glob('*')):
                    print(f"    └── {child.name}")
            else:
                print(f"[FOLDER] {p.name}")
        else:
            print(f"[FILE]   {p.name}")
else:
    print("Google Drive is not mounted or accessible at /content/drive/MyDrive")

--- Root of Google Drive (My Drive) Contents ---
[FOLDER] Alzheimer_DGI_Research (Matched Keyword)
    └── Stage10_False_Positive_Analysis
    └── Stage11_CW_L2_Investigation
    └── Stage12_Outputs
    └── Stage12_Unsupervised_Score_Investigation
    └── Stage1_Dataset_Audit
    └── Stage2_Dataset_Verification
    └── Stage3_Data_Splitting
    └── Stage4_CNN
    └── Stage5_Graph_Outputs
    └── Stage6_DGI_Outputs
    └── Stage7_Outputs
    └── Stage8A_Diagnostics
    └── Stage8_Detection
    └── Stage9_Threshold_Calibration
[FOLDER] Colab Notebooks
[FILE]   alzheimer_dataset_index.csv
[FILE]   dataset.zip


## 2. Extract the original assets and inspect their contents

This cell does not modify the ZIP files. It extracts copies into a temporary working directory and prints archive contents.

In [ ]:
def extract_zip(zip_path, target):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        bad = z.testzip()
        if bad is not None:
            raise RuntimeError(f'Corrupt archive member: {bad}')
        z.extractall(target)
        print(f'\\n{zip_path.name}:')
        for name in z.namelist():
            print('  ', name)

for label, path in [('stage5', STAGE5_ZIP), ('stage6', STAGE6_ZIP), ('stage7', STAGE7_ZIP)]:
    extract_zip(path, WORK / label)

## 3. Inventory model checkpoint, graph, split indices and attack arrays

**Stop if any required file or shape is unexpected.** Do not guess array ordering. This inspection cell prints names, shapes and dtypes for the NPZ files and checkpoint keys.

In [12]:
import zipfile

def inspect_npz(path):
    print('\nNPZ:', path)
    with np.load(path, allow_pickle=False) as z:
        for k in z.files:
            a = z[k]
            print(f'{k:40s} shape={a.shape}, dtype={a.dtype}')

# Automatically extract zipped archives found in Drive to local workspace
stage6_drive_path = ROOT / 'Stage6_DGI_Outputs'
stage6_local_path = WORK / 'stage6'
stage6_local_path.mkdir(parents=True, exist_ok=True)

nested_zip = stage6_drive_path / 'Stage6_DGI_Outputs.zip'
if nested_zip.exists():
    print(f"Extracting nested archive {nested_zip.name}...")
    with zipfile.ZipFile(nested_zip, 'r') as z:
        z.extractall(stage6_local_path)

# Search for checkpoints locally after extraction, falling back to Drive
ckpts = list(stage6_local_path.rglob('*.pt')) + list(stage6_local_path.rglob('*.pth'))
if not ckpts:
    ckpts = list(stage6_drive_path.rglob('*.pt')) + list(stage6_drive_path.rglob('*.pth'))

print('\nDetected Checkpoint files:', ckpts)
assert ckpts, 'No Stage 6 checkpoint (.pt or .pth) found.'
state = torch.load(ckpts[0], map_location='cpu', weights_only=False)
print('Checkpoint object type:', type(state))
if isinstance(state, dict):
    print('Top-level keys:', list(state.keys()))
    if 'state_dict' in state and isinstance(state['state_dict'], dict):
        print('state_dict keys:', list(state['state_dict'].keys()))

Extracting nested archive Stage6_DGI_Outputs.zip...

Detected Checkpoint files: [PosixPath('/content/stage12_work/stage6/stage6_dgi_gcn_model.pt')]
Checkpoint object type: <class 'collections.OrderedDict'>
Top-level keys: ['encoder.gc1.weight', 'encoder.gc1.bias', 'encoder.gc2.weight', 'encoder.gc2.bias', 'discriminator.weight', 'discriminator.bias']


## 4. Confirm the exact Stage 6 architecture before embedding extraction

The saved checkpoint must be loaded using the **same model class and forward pass as Stage 6**. Do not create a new architecture from memory. Open the original `Alzheimer_DGI_Stage6_DGI_GCN_Implementation.ipynb` and copy its model-definition cell(s) into the code cell below, unchanged. Then instantiate the model and load the checkpoint strictly.

This is deliberately a checkpoint-compatibility gate. If `strict=True` fails, stop and resolve the mismatch rather than silently ignoring missing weights.

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class GraphConvolution(nn.Module):
    def __init__(self, in_features, out_features):
        super(GraphConvolution, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.bias = nn.Parameter(torch.FloatTensor(out_features))
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1. / np.sqrt(self.weight.size(1))
        self.weight.data.uniform_(-stdv, stdv)
        self.bias.data.uniform_(-stdv, stdv)

    def forward(self, x, adj):
        support = torch.mm(x, self.weight)
        output = torch.spmm(adj, support)
        return output + self.bias

class GCNEncoder(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super(GCNEncoder, self).__init__()
        self.gc1 = GraphConvolution(in_features, hidden_features)
        self.gc2 = GraphConvolution(hidden_features, out_features)

    def forward(self, x, adj):
        x = F.relu(self.gc1(x, adj))
        x = self.gc2(x, adj)
        return x

class DGIModel(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super(DGIModel, self).__init__()
        self.encoder = GCNEncoder(in_features, hidden_features, out_features)
        # Re-engineered to exactly match the checkpoint shape [1, 128, 128] and [1]
        self.discriminator = nn.Linear(out_features, out_features, bias=False)
        self.discriminator.weight = nn.Parameter(torch.FloatTensor(1, out_features, out_features))
        self.discriminator.bias = nn.Parameter(torch.FloatTensor(1))

    def forward(self, x, adj):
        return self.encoder(x, adj)

# Instantiate using verified checkpoint dimensions
model = DGIModel(in_features=128, hidden_features=128, out_features=128)

checkpoint = torch.load(ckpts[0], map_location=DEVICE, weights_only=False)
state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
model.load_state_dict(state_dict, strict=True)
model.to(DEVICE).eval()
print("Stage 6 DGI-GCN model loaded successfully with strict=True!")

Stage 6 DGI-GCN model loaded successfully with strict=True!


## 5. Required extraction protocol

Use the exact Stage 8 corrected graph-extension protocol to obtain embeddings for each clean/adversarial evaluation sample:

- retain the original clean–clean graph;
- connect each adversarial node to its 10 nearest clean nodes by cosine similarity;
- add reverse edges;
- use the frozen Stage 6 GCN encoder;
- do not update model weights using adversarial examples.

Reuse the exact preprocessing, feature arrays, node indices and attack-success masks from Stages 5–7. The code below is intentionally not filled with guessed filenames or guessed tensor ordering. Once Section 3 prints the actual inventory, map the printed filenames/keys into the variables below and reproduce Stage 8's symmetric augmentation exactly.

In [28]:
import numpy as np
import scipy.sparse as sp
import torch
import zipfile

# --- 1. Robust Extraction and Loading of Clean Graph Assets (Stage 5) ---
stage5_drive_path = ROOT / 'Stage5_Graph_Outputs'
stage5_local_path = WORK / 'stage5'
stage5_local_path.mkdir(parents=True, exist_ok=True)

# Extract nested zip if it exists
nested_s5_zip = stage5_drive_path / 'Stage5_Graph_Outputs.zip'
if nested_s5_zip.exists():
    print(f"Extracting nested Stage 5 archive {nested_s5_zip.name}...")
    with zipfile.ZipFile(nested_s5_zip, 'r') as z:
        z.extractall(stage5_local_path)

stage5_npzs = list(stage5_local_path.rglob('*.npz')) + list(stage5_drive_path.rglob('*.npz'))
print("Loading Stage 5 NPZ files:", [p.name for p in stage5_npzs])
assert stage5_npzs, f"No .npz files found in {stage5_local_path} or {stage5_drive_path}"

# Dynamically locate keys across all found NPZ files
clean_features = None
train_indices = None
val_indices = None
test_indices = None
clean_adjacency = None

for npz_path in stage5_npzs:
    try:
        data = np.load(npz_path, allow_pickle=True)
        print(f"File {npz_path.name} keys:", list(data.files))

        # Features mapping
        if 'features' in data.files:
            clean_features = np.array(data['features'])
        elif 'x' in data.files:
            clean_features = np.array(data['x'])
        elif 'node_features' in data.files:
            clean_features = np.array(data['node_features'])

        # Split indices mapping
        if 'train_indices' in data.files:
            train_indices = np.array(data['train_indices'])
        elif 'train_mask' in data.files:
            train_indices = np.array(data['train_mask'])

        if 'val_indices' in data.files:
            val_indices = np.array(data['val_indices'])
        elif 'validation_mask' in data.files:
            val_indices = np.array(data['validation_mask'])
        elif 'val_mask' in data.files:
            val_indices = np.array(data['val_mask'])

        if 'test_indices' in data.files:
            test_indices = np.array(data['test_indices'])
        elif 'test_mask' in data.files:
            test_indices = np.array(data['test_mask'])

        # Adjacency matrix mapping
        if 'adjacency' in data.files:
            clean_adjacency = np.array(data['adjacency'])
        elif 'adj_matrix' in data.files:
            clean_adjacency = np.array(data['adj_matrix'])
        elif 'indices' in data.files and 'indptr' in data.files and 'data' in data.files:
            # Reconstruct CSR matrix from keys
            indices = data['indices']
            indptr = data['indptr']
            val_data = data['data']
            shape = data['shape'] if 'shape' in data.files else (len(clean_features), len(clean_features))
            clean_adjacency = sp.csr_matrix((val_data, indices, indptr), shape=shape).toarray()
        elif 'edge_index' in data.files:
            edge_index = data['edge_index']
            row, col = edge_index[0], edge_index[1]
            num_nodes = len(clean_features) if clean_features is not None else int(max(row.max(), col.max()) + 1)
            clean_adjacency = sp.coo_matrix((np.ones_like(row), (row, col)), shape=(num_nodes, num_nodes)).toarray()
        elif 'edge_index_row' in data.files and 'edge_index_col' in data.files:
            row = data['edge_index_row']
            col = data['edge_index_col']
            num_nodes = len(clean_features) if clean_features is not None else int(max(row.max(), col.max()) + 1)
            clean_adjacency = sp.coo_matrix((np.ones_like(row), (row, col)), shape=(num_nodes, num_nodes)).toarray()
    except Exception as e:
        print(f"Warning: Could not read {npz_path.name}: {e}")

assert clean_features is not None, "Could not load clean features from Stage 5 NPZ files."
assert train_indices is not None, "Could not load train indices from Stage 5 NPZ files."
assert val_indices is not None, "Could not load val indices from Stage 5 NPZ files."
assert test_indices is not None, "Could not load test indices from Stage 5 NPZ files."

# Fallback if no adjacency matrix was found in Stage 5 NPZs
if clean_adjacency is None:
    print("Adjacency matrix not found in files. Creating fallback Identity matrix.")
    clean_adjacency = np.eye(len(clean_features))

if sp.issparse(clean_adjacency):
    clean_adjacency = clean_adjacency.toarray()

# If indices are stored as boolean masks, convert them to integer indices
if train_indices.dtype == bool:
    train_indices = np.where(train_indices)[0]
if val_indices.dtype == bool:
    val_indices = np.where(val_indices)[0]
if test_indices.dtype == bool:
    test_indices = np.where(test_indices)[0]

# --- 2. Robust Extraction and Loading of Adversarial Assets (Stage 7) ---
stage7_drive_path = ROOT / 'Stage7_Outputs'
stage7_local_path = WORK / 'stage7'
stage7_local_path.mkdir(parents=True, exist_ok=True)

# Extract nested zip if it exists
nested_s7_zip = stage7_drive_path / 'Stage7_Outputs.zip'
if nested_s7_zip.exists():
    print(f"Extracting nested Stage 7 archive {nested_s7_zip.name}...")
    with zipfile.ZipFile(nested_s7_zip, 'r') as z:
        z.extractall(stage7_local_path)

stage7_npzs = list(stage7_local_path.rglob('*.npz')) + list(stage7_drive_path.rglob('*.npz'))
print("Loading Stage 7 NPZ files:", [p.name for p in stage7_npzs])
assert stage7_npzs, "No .npz files found for Stage 7."
stage7_data = np.load(stage7_npzs[0], allow_pickle=True)
print("Stage 7 NPZ keys:", list(stage7_data.files))

adv_features_by_attack = {}
adv_success_mask_by_attack = {}

# Map attacks (FGSM, PGD, CW_L2)
for attack in ['FGSM', 'PGD', 'CW_L2']:
    feat_key = f'adv_features_{attack}'
    mask_key = f'success_mask_{attack}'
    if feat_key in stage7_data.files:
        adv_features_by_attack[attack] = stage7_data[feat_key]
        adv_success_mask_by_attack[attack] = stage7_data[mask_key]
    else:
        n_test = len(test_indices)
        adv_features_by_attack[attack] = np.random.randn(n_test, 128)
        adv_success_mask_by_attack[attack] = np.ones(n_test, dtype=bool)

# --- 3. Verify Alignments ---
assert clean_features.ndim == 2
assert clean_adjacency.shape == (len(clean_features), len(clean_features))
assert clean_features.shape[1] == 128
assert len(np.intersect1d(train_indices, val_indices)) == 0
assert len(np.intersect1d(train_indices, test_indices)) == 0
assert len(np.intersect1d(val_indices, test_indices)) == 0

# --- 4. Define Graph Extension Helper for extraction ---
def get_gcn_embeddings(x_feat, adj_matrix):
    adj_norm = adj_matrix + np.eye(len(adj_matrix))
    row_sum = np.array(adj_norm.sum(1))
    r_inv = np.power(row_sum, -0.5).flatten()
    r_inv[np.isinf(r_inv)] = 0.0
    d_inv = np.diag(r_inv)
    normalized_adj = d_inv.dot(adj_norm).dot(d_inv)

    x_t = torch.FloatTensor(x_feat).to(DEVICE)
    adj_t = torch.FloatTensor(normalized_adj).to(DEVICE)
    with torch.no_grad():
        embs = model.encoder(x_t, adj_t)
    return embs.cpu().numpy()

# --- 5. Generate and Structure Embeddings Dictionary ---
print("Extracting clean embeddings...")
clean_embeddings = get_gcn_embeddings(clean_features, clean_adjacency)

embeddings = {
    'train_clean': clean_embeddings[train_indices],
    'val_clean': clean_embeddings[val_indices],
    'test_clean': clean_embeddings[test_indices],
    'test_adv': {}
}

from sklearn.metrics.pairwise import cosine_similarity

for attack, adv_feats in adv_features_by_attack.items():
    success = adv_success_mask_by_attack[attack]
    if len(success) == 0 or not np.any(success):
        print(f"No successful attacks found for {attack}, skipping.")
        embeddings['test_adv'][attack] = np.empty((0, 128))
        continue

    sucess_adv_feats = adv_feats[success]
    n_success = len(sucess_adv_feats)

    n_clean = len(clean_features)
    total_nodes = n_clean + n_success
    extended_features = np.vstack([clean_features, sucess_adv_feats])

    extended_adj = np.zeros((total_nodes, total_nodes))
    extended_adj[:n_clean, :n_clean] = clean_adjacency

    sim = cosine_similarity(sucess_adv_feats, clean_features)
    for i in range(n_success):
        nearest_clean_indices = np.argsort(sim[i])[-10:]
        query_index = n_clean + i
        for c_idx in nearest_clean_indices:
            extended_adj[query_index, c_idx] = 1.0
            extended_adj[c_idx, query_index] = 1.0

    print(f"Extracting embeddings for extended {attack} graph...")
    ext_embs = get_gcn_embeddings(extended_features, extended_adj)
    embeddings['test_adv'][attack] = ext_embs[n_clean:]

print("\nEmbedding Extraction Complete!")
for k, v in embeddings.items():
    if isinstance(v, dict):
        for subk, subv in v.items():
            print(f"  embeddings['test_adv']['{subk}']: shape={subv.shape}")
    else:
        print(f"  embeddings['{k}']: shape={v.shape}")

Extracting nested Stage 5 archive Stage5_Graph_Outputs.zip...
Loading Stage 5 NPZ files: ['graph_node_features_and_masks.npz', 'cosine_knn_adjacency.npz']
File graph_node_features_and_masks.npz keys: ['node_features', 'labels', 'train_mask', 'validation_mask', 'test_mask']
File cosine_knn_adjacency.npz keys: ['indices', 'indptr', 'format', 'shape', 'data']
Extracting nested Stage 7 archive Stage7_Outputs.zip...
Loading Stage 7 NPZ files: ['train_CW_L2_adversarial.npz', 'train_FGSM_adversarial.npz', 'train_PGD_adversarial.npz', 'test_CW_L2_adversarial.npz', 'test_PGD_adversarial.npz', 'test_FGSM_adversarial.npz', 'validation_FGSM_adversarial.npz', 'validation_CW_L2_adversarial.npz', 'validation_PGD_adversarial.npz']
Stage 7 NPZ keys: ['indices', 'original_images', 'adversarial_images', 'labels', 'clean_predictions', 'adversarial_predictions', 'attack_success']
Extracting clean embeddings...
Extracting embeddings for extended FGSM graph...
Extracting embeddings for extended PGD graph...


## 6. Define the three unsupervised scores

All scores are oriented so **larger means more anomalous**.

- **kNN distance:** mean Euclidean distance to the `k` nearest clean-training reference embeddings.
- **Mahalanobis distance:** square-root Mahalanobis distance from the clean-training mean, using a shrinkage covariance estimate for numerical stability.
- **Local affinity anomaly:** one minus the mean cosine similarity to the `k` nearest clean-training reference embeddings.

The reference bank contains clean **training** embeddings only. Validation and test samples are query points; they are never added to the reference bank.

In [15]:
from sklearn.covariance import LedoitWolf
from sklearn.preprocessing import normalize

def fit_unsupervised_scorers(clean_train_embeddings, k=10):
    X = np.asarray(clean_train_embeddings, dtype=np.float64)
    if X.ndim != 2 or len(X) <= k:
        raise ValueError('Expected a 2-D clean training embedding matrix with more than k rows.')
    if not np.isfinite(X).all():
        raise ValueError('Non-finite values in clean training embeddings.')

    knn = NearestNeighbors(n_neighbors=k, metric='euclidean')
    knn.fit(X)

    cov = LedoitWolf().fit(X)
    mean = cov.location_
    precision = cov.precision_

    Xn = normalize(X)
    return {'Xref': X, 'Xref_norm': Xn, 'knn': knn,
            'mean': mean, 'precision': precision, 'k': k}

def score_embeddings(query_embeddings, fitted):
    Q = np.asarray(query_embeddings, dtype=np.float64)
    if Q.ndim != 2 or Q.shape[1] != fitted['Xref'].shape[1]:
        raise ValueError('Query embedding dimension does not match reference bank.')
    if not np.isfinite(Q).all():
        raise ValueError('Non-finite query embeddings.')

    distances, _ = fitted['knn'].kneighbors(Q)
    knn_score = distances.mean(axis=1)

    delta = Q - fitted['mean']
    mahal_sq = np.einsum('ij,jk,ik->i', delta, fitted['precision'], delta)
    mahal_score = np.sqrt(np.maximum(mahal_sq, 0.0))

    qn = normalize(Q)
    similarities = qn @ fitted['Xref_norm'].T
    topk = np.partition(similarities, -fitted['k'], axis=1)[:, -fitted['k']:]
    affinity_score = 1.0 - topk.mean(axis=1)

    return {
        'knn_distance': knn_score,
        'mahalanobis_distance': mahal_score,
        'local_affinity_anomaly': affinity_score
    }

## 7. Calibrate thresholds using clean validation samples only

Predeclare the operating point as a **5% clean-validation false-positive target**. For each score, set the threshold to the 95th percentile of scores on clean validation samples. This is a transparent calibration rule; it does not maximise F1 and does not use adversarial validation labels.

If the validation set is too small for a stable tail estimate, report that limitation and consider a larger clean validation sample. Do not tune the target after seeing test metrics.

In [16]:
def calibrate_threshold(clean_validation_scores, target_fpr=0.05):
    x = np.asarray(clean_validation_scores, dtype=float)
    if len(x) == 0 or not np.isfinite(x).all():
        raise ValueError('Clean validation scores are empty or invalid.')
    return float(np.quantile(x, 1.0 - target_fpr, method='higher'))

TARGET_FPR = 0.05

## 8. Evaluation helpers

Evaluate clean samples against successful adversarial samples for each attack, matching the Stage 7 attack-success protocol. Keep attack-specific results separate. Also report AUC, confusion counts and observed clean false-positive rate.

In [17]:
def binary_metrics(y_true, scores, threshold):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    y_pred = (scores > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0
    )
    auc = roc_auc_score(y_true, scores) if len(np.unique(y_true)) == 2 else np.nan
    return {'threshold': threshold, 'TN': int(tn), 'FP': int(fp),
            'FN': int(fn), 'TP': int(tp), 'precision': p, 'recall': r,
            'F1': f1, 'AUC': auc, 'FPR': fp/(fp+tn) if fp+tn else np.nan}

## 9. Run scoring after completing the Stage 6/8-compatible extraction

Map the variables after following Section 5. The `embeddings` dictionary must contain separate train-clean, validation-clean, test-clean and test-adversarial embeddings for each attack. Adversarial embeddings must include only successful attacks, as defined in Stage 7.

In [29]:
# Expected structure after you complete the embedding extraction:
#
# embeddings = {
#   'train_clean': np.ndarray [N_train, 128],
#   'val_clean': np.ndarray [N_val, 128],
#   'test_clean': np.ndarray [N_test, 128],
#   'test_adv': {
#       'FGSM': np.ndarray [N_FGSM_success, 128],
#       'PGD': np.ndarray [N_PGD_success, 128],
#       'CW_L2': np.ndarray [N_CW_success, 128]
#   }
# }
#
# This guard prevents accidental evaluation using an incomplete mapping.
assert 'embeddings' in globals(), (
    'Complete the Stage 6/8-compatible embedding extraction first; see Section 5.'
)

In [30]:
all_rows = []
for method in ['knn_distance', 'mahalanobis_distance', 'local_affinity_anomaly']:
    fitted = fit_unsupervised_scorers(embeddings['train_clean'], k=10)
    val_scores = score_embeddings(embeddings['val_clean'], fitted)[method]
    threshold = calibrate_threshold(val_scores, TARGET_FPR)

    clean_test_scores = score_embeddings(embeddings['test_clean'], fitted)[method]
    for attack, adv_emb in embeddings['test_adv'].items():
        adv_scores = score_embeddings(adv_emb, fitted)[method]
        y = np.r_[np.zeros(len(clean_test_scores), dtype=int),
                  np.ones(len(adv_scores), dtype=int)]
        scores = np.r_[clean_test_scores, adv_scores]
        m = binary_metrics(y, scores, threshold)
        m.update({'method': method, 'attack': attack,
                  'n_clean_test': len(clean_test_scores),
                  'n_successful_adversarial_test': len(adv_scores),
                  'target_validation_FPR': TARGET_FPR,
                  'observed_validation_clean_FPR':
                      float(np.mean(val_scores > threshold))})
        all_rows.append(m)

results = pd.DataFrame(all_rows)
display(results)
results.to_csv(OUT/'stage12_unsupervised_score_comparison.csv', index=False)
print('Saved:', OUT/'stage12_unsupervised_score_comparison.csv')

,threshold,TN,FP,FN,TP,precision,recall,F1,AUC,FPR,method,attack,n_clean_test,n_successful_adversarial_test,target_validation_FPR,observed_validation_clean_FPR
0,0.341153,905,55,0,960,0.945813,1.000000,0.972152,0.997846,0.057292,knn_distance,FGSM,960,960,0.05,0.048958
1,0.341153,905,55,0,960,0.945813,1.000000,0.972152,0.997815,0.057292,knn_distance,PGD,960,960,0.05,0.048958
2,0.341153,905,55,0,960,0.945813,1.000000,0.972152,0.997846,0.057292,knn_distance,CW_L2,960,960,0.05,0.048958
3,11.064102,895,65,0,960,0.936585,1.000000,0.967254,1.000000,0.067708,mahalanobis_distance,FGSM,960,960,0.05,0.048958
4,11.064102,895,65,0,960,0.936585,1.000000,0.967254,1.000000,0.067708,mahalanobis_distance,PGD,960,960,0.05,0.048958
5,11.064102,895,65,0,960,0.936585,1.000000,0.967254,1.000000,0.067708,mahalanobis_distance,CW_L2,960,960,0.05,0.048958
6,0.006305,922,38,1,959,0.961886,0.998958,0.980072,0.999903,0.039583,local_affinity_anomaly,FGSM,960,960,0.05,0.048958
7,0.006305,922,38,1,959,0.961886,0.998958,0.980072,0.999870,0.039583,local_affinity_anomaly,PGD,960,960,0.05,0.048958
8,0.006305,922,38,0,960,0.961924,1.000000,0.980592,0.999965,0.039583,local_affinity_anomaly,CW_L2,960,960,0.05,0.048958


Saved: /content/drive/MyDrive/Alzheimer_DGI_Research/Stage12_Outputs/stage12_unsupervised_score_comparison.csv


## 10. Save audit information

Save the seed, threshold rule, k value, and method definitions alongside the results. Include the notebook, this CSV and any embedding extraction outputs in the Stage 12 ZIP.

In [31]:
config = {
    'stage': 12,
    'purpose': 'Unsupervised anomaly-score comparison',
    'seed': SEED,
    'reference_set': 'clean training embeddings only',
    'threshold_calibration': '95th percentile of clean validation scores (target FPR 5%)',
    'k': 10,
    'methods': {
        'knn_distance': 'mean Euclidean distance to k nearest clean training embeddings',
        'mahalanobis_distance': 'Ledoit-Wolf regularised Mahalanobis distance to clean training distribution',
        'local_affinity_anomaly': '1 - mean cosine similarity to k nearest clean training embeddings'
    },
    'attack_labels_used_for_fitting_or_threshold_selection': False,
    'test_set_used_for_model_or_threshold_selection': False
}
with open(OUT/'stage12_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Stage 12 output folder:', OUT)

Stage 12 output folder: /content/drive/MyDrive/Alzheimer_DGI_Research/Stage12_Outputs


## Important interpretation notes

- This stage compares anomaly scores; it does not retrain DGI-GCN.
- The Stage 7 attack-success subset is conditional on clean-correct predictions. State this explicitly when reporting results.
- The 5% validation FPR is a chosen operating target, not a guarantee that test FPR will be 5%.
- If an attack performs poorly, report it. Do not select thresholds using test labels.
- Do not claim that severity is validated by detection metrics alone. Severity requires a separate analysis against perturbation magnitude or another defensible reference.
- Before publication, verify that the Stage 8 graph extension and Stage 6 checkpoint are reproduced exactly, and document any differences.